# SongForge-DL Remote Colab

Run this notebook in Google Colab from the Google account that should own the Drive files. It keeps large downloads and training on Colab/Drive, not on the local workstation.

In [ ]:
REPO_URL = "https://github.com/auth889-ai/ml-sing.git"
BRANCH = "main"
PROJECT_SUBDIR = "songforge-dl-starter"
DRIVE_ROOT = "/content/drive/MyDrive/songforge-dl"
SONGFORGE_DATA = f"{DRIVE_ROOT}/data"
ACCEPT_NONCOMMERCIAL_DATASET_TERMS = False
DATASET_ID = "babyslakh"  # babyslakh, slakh2100, lakh_midi, gtsinger, mtg_jamendo, nsynth


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "$DRIVE_ROOT" "$SONGFORGE_DATA/raw" "$SONGFORGE_DATA/processed" "$SONGFORGE_DATA/manifests" "$DRIVE_ROOT/checkpoints" "$DRIVE_ROOT/logs"
!nvidia-smi || true


In [ ]:
import os
repo_dir = f"{DRIVE_ROOT}/repo"
if not os.path.exists(repo_dir):
    !git clone --branch "$BRANCH" "$REPO_URL" "$repo_dir"
else:
    %cd "$repo_dir"
    !git fetch origin "$BRANCH"
    !git checkout "$BRANCH"
    !git pull --ff-only origin "$BRANCH"
%cd "$repo_dir/$PROJECT_SUBDIR"
!pip install -q -e '.[dev,audio]'
!python scripts/validate_dataset_registry.py
!pytest -q


In [ ]:
import os, subprocess, yaml
os.environ['SONGFORGE_DATA'] = SONGFORGE_DATA
with open('configs/data/datasets.yaml', 'r', encoding='utf-8') as f:
    registry = yaml.safe_load(f)
spec = registry['datasets'][DATASET_ID]
needs_acceptance = spec['license'].get('requires_user_acceptance') or spec['access'].get('gated')
print(f"Selected: {DATASET_ID} - {spec['name']}")
print(f"License: {spec['license']['name']}")
print(f"Estimated size: {spec['access'].get('estimated_size')}")
if needs_acceptance and not ACCEPT_NONCOMMERCIAL_DATASET_TERMS:
    raise SystemExit('Set ACCEPT_NONCOMMERCIAL_DATASET_TERMS=True only after reading and accepting upstream dataset terms.')
for command in spec['access'].get('colab_commands', []):
    print('RUN:', command)
    subprocess.run(command, shell=True, check=True)


## Training Gate

The repository contract requires M01 and M02 before full model training. At the current phase, run registry validation and tests. When M02/M03 trainers are implemented, replace the guarded commands below with the milestone trainer command.

In [ ]:
!python scripts/validate_dataset_registry.py
# Future M03 example after implementation:
# !python scripts/train_codec.py --config configs/codec/codec_small.yaml --data-root "$SONGFORGE_DATA" --checkpoint-root "$DRIVE_ROOT/checkpoints"
